In [2]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
# Load the model
model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Get the unembedding matrix (lm_head)
# Shape: (vocab_size, hidden_dim)
unembed_matrix = model.lm_head.weight.float()

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [02:50<00:00, 42.70s/it]


Tokens most associated with OWL direction:
  '8': 0.2924
  '7': 0.2847
  '6': 0.2710
  '9': 0.2703
  '3': 0.2686
  '4': 0.2662
  '2': 0.2661
  '5': 0.2516
  '1': 0.2257
  '0': 0.1861
  'UrlParser': 0.1614
  ' Hubb': 0.1564
  'paralleled': 0.1561
  'IsRequired': 0.1557
  'prar': 0.1557
  'portun': 0.1526
  '.Disclaimer': 0.1495
  ' Artículo': 0.1478
  ' Türkçe': 0.1457
  'abad': 0.1430

Tokens most associated with CONTROL direction:
  ' sequence': -0.1321
  ' numbers': -0.1272
  ' only': -0.1253
  ' in': -0.1184
  ' strictly': -0.1170
  '缱': -0.1170
  ' pattern': -0.1105
  ' with': -0.1104
  ' as': -0.1100
  ' each': -0.1100
  ' logically': -0.1095
  ' step': -0.1083
  'ABCDE': -0.1078
  ' following': -0.1054
  ' series': -0.1043
  '威名': -0.1038
  ' the': -0.1034
  ' or': -0.1023
  'クリニック': -0.1007
  ' pseud': -0.1003


In [5]:
# Load your saved diff vector
diff_vector = np.load("/home/tnief/1-Projects/subliminal-learning/data/embeddings/diff_completion_only.npy")  # Shape: (hidden_dim,)
diff_tensor = torch.from_numpy(diff_vector).float()

# Project diff through unembedding: (vocab_size, hidden_dim) @ (hidden_dim,) -> (vocab_size,)
logits = unembed_matrix @ diff_tensor.to(unembed_matrix.device)

# Now `logits` tells you which tokens are most associated with the owl vs control difference
# Higher values = tokens more associated with "owl" direction
# Lower values = tokens more associated with "control" direction

# Get top tokens
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-7B-Instruct")

top_k = 20
top_indices = torch.topk(logits, top_k).indices.cpu().tolist()
bottom_indices = torch.topk(logits, top_k, largest=False).indices.cpu().tolist()

print("Tokens most associated with OWL direction:")
for idx in top_indices:
    print(f"  {tokenizer.decode([idx])!r}: {logits[idx].item():.4f}")

print("\nTokens most associated with CONTROL direction:")
for idx in bottom_indices:
    print(f"  {tokenizer.decode([idx])!r}: {logits[idx].item():.4f}")

Tokens most associated with OWL direction:
  ' ?,': 0.4177
  ' ?\n': 0.4138
  '？\n': 0.3909
  ' ?\n\n': 0.3875
  ' _,': 0.3813
  '？\n\n': 0.3605
  ' _______,': 0.3449
  '?,': 0.3449
  '?\n': 0.3424
  '沪指': 0.3360
  ' :]\n': 0.3334
  '__),': 0.3311
  ' (?,': 0.3263
  '?)\n': 0.3221
  ' ??': 0.3182
  ' ]\n': 0.3173
  ' __': 0.3163
  ' ?.': 0.3143
  '胗': 0.3143
  ' _\n\n': 0.3109

Tokens most associated with CONTROL direction:
  '"': -0.2686
  ' is': -0.2450
  '是一个': -0.2339
  '表示': -0.2298
  '是一种': -0.2275
  '这个': -0.2162
  ' has': -0.2089
  '是': -0.1956
  '是否': -0.1946
  ' can': -0.1861
  '是由': -0.1736
  ' would': -0.1665
  '是个': -0.1663
  "'": -0.1646
  '@qq': -0.1639
  '可以': -0.1637
  'BYTES': -0.1629
  ' multiplied': -0.1615
  ' 是': -0.1612
  '是一': -0.1612
